# 01 - NumPy 基础：AI 计算的数组语言

NumPy 是 Python 科学计算生态的底座。机器学习里常见的张量、批量样本、特征矩阵、权重矩阵、激活函数，本质上都可以先用 NumPy 的 `ndarray` 来理解。

## 本 Notebook 概览

本节不是把 `.py` 文件逐行搬进 Notebook，而是围绕“为什么 NumPy 适合 AI 计算”重新组织：

| 章节 | 你会学到什么 | 和 AI 的关系 |
|---|---|---|
| 1. ndarray 的心智模型 | shape、ndim、dtype、连续存储 | 张量形状调试的基础 |
| 2. 创建数组 | zeros、ones、arange、linspace、随机数 | 构造数据、初始化参数 |
| 3. 索引与切片 | 基本索引、布尔索引、花式索引 | 取 batch、筛选样本、处理特征 |
| 4. 向量化 | 用数组运算替代 Python 循环 | 速度提升的关键 |
| 5. 广播机制 | 不同形状如何自动对齐 | bias、标准化、批量计算 |
| 6. 聚合与形状操作 | axis、reshape、transpose、拼接 | 批量统计和神经网络维度变换 |
| 7. 线性代数与 AI 小实验 | 矩阵乘法、激活函数、softmax | 从数组到一个极简模型 |
| 8. 检查清单 | 常见错误与调试方法 | 减少 shape bug |


In [ ]:
# 环境准备
import numpy as np
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

plt.style.use("seaborn-v0_8-whitegrid")
print("NumPy 版本:", np.__version__)


## 1. ndarray 的心智模型：值 + 形状 + 类型

Python 列表像“装着对象引用的盒子”；NumPy 数组更像“一整块连续内存 + 一套解释规则”。

理解一个数组时，先看三个属性：

- `shape`：每个轴有多长，例如 `(3, 4)` 表示 3 行 4 列
- `ndim`：轴的个数，例如矩阵是 2 维，图片批次可能是 4 维
- `dtype`：每个元素的数据类型，例如 `float64`、`int32`

在 AI 中，很多错误不是数学错，而是“形状对不上”。所以从第一天起就要养成打印 shape 的习惯。


In [ ]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([[1, 2, 3],
              [4, 5, 6]])

for name, arr in [("a", a), ("b", b)]:
    print(f"{name} =\n{arr}")
    print(f"  shape: {arr.shape}")
    print(f"  ndim : {arr.ndim}")
    print(f"  dtype: {arr.dtype}")
    print(f"  size : {arr.size} 个元素")
    print()


In [ ]:
# 可视化一个 3x4 数组：每个格子是一个元素，每个轴有自己的含义
mat = np.arange(12).reshape(3, 4)
fig, ax = plt.subplots(figsize=(6, 3))
im = ax.imshow(mat, cmap="Blues")
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, str(mat[i, j]), ha="center", va="center", fontsize=12)
ax.set_title("一个 shape=(3, 4) 的二维 ndarray")
ax.set_xlabel("axis=1：列 / 特征")
ax.set_ylabel("axis=0：行 / 样本")
ax.set_xticks(range(4))
ax.set_yticks(range(3))
plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()


### 1.1 dtype 为什么重要？

`dtype` 决定了每个元素占多少内存，也影响数值精度。

- `float64`：默认浮点类型，精度高，内存占用较大
- `float32`：深度学习里常用，速度和显存更友好
- `int64`：默认整数类型，适合索引、计数、类别编号

一个 100 万个元素的 `float64` 数组约占 8 MB，而 `float32` 约占 4 MB。


In [ ]:
arr64 = np.ones(1_000_000, dtype=np.float64)
arr32 = np.ones(1_000_000, dtype=np.float32)

print("float64 数组内存:", arr64.nbytes / 1024**2, "MB")
print("float32 数组内存:", arr32.nbytes / 1024**2, "MB")
print("节省比例:", arr32.nbytes / arr64.nbytes)


## 2. 创建数组：从“占位”到“参数初始化”

NumPy 提供了很多创建数组的方式。它们不是孤立 API，而对应常见任务：

| 函数 | 用途 | 典型场景 |
|---|---|---|
| `np.array` | 从 Python 数据创建 | 小样例、手写矩阵 |
| `np.zeros` | 全 0 占位 | 初始化累加器、mask |
| `np.ones` | 全 1 | 构造偏置、测试广播 |
| `np.eye` | 单位矩阵 | 线性代数、保持原向量不变 |
| `np.arange` | 固定步长序列 | 索引、离散坐标 |
| `np.linspace` | 固定数量等距点 | 画函数曲线 |
| `rng.normal` | 正态随机数 | 模拟数据、权重初始化 |


In [ ]:
print("zeros:\n", np.zeros((2, 3)))
print("\nones:\n", np.ones((2, 3)))
print("\neye:\n", np.eye(4))
print("\narange:", np.arange(0, 10, 2))
print("linspace:", np.linspace(0, 1, 5))


In [ ]:
# linspace 常用于画连续函数
xs = np.linspace(-2*np.pi, 2*np.pi, 400)
ys_sin = np.sin(xs)
ys_cos = np.cos(xs)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(xs, ys_sin, label="sin(x)")
ax.plot(xs, ys_cos, label="cos(x)")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("np.linspace：为函数曲线生成均匀采样点")
ax.legend()
plt.show()


### 2.1 随机数与可复现实验

机器学习实验经常需要随机初始化、随机划分数据、随机打乱 batch。为了让结果可复现，推荐使用新的随机数生成器：

```python
rng = np.random.default_rng(42)
```

其中 `42` 是随机种子。相同种子会生成相同随机序列，便于调试和写教程。


In [ ]:
rng = np.random.default_rng(42)
uniform = rng.random((3, 4))          # [0, 1) 均匀分布
normal = rng.normal(0, 1, (3, 4))     # 均值 0、标准差 1 的正态分布
integers = rng.integers(0, 10, size=8)

print("均匀分布:\n", uniform)
print("\n正态分布:\n", normal)
print("\n随机整数:", integers)


In [ ]:
# 可视化两种常见分布
rng = np.random.default_rng(42)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].hist(rng.random(10_000), bins=40, color="#4C78A8", alpha=0.85)
axes[0].set_title("均匀分布 U(0, 1)")
axes[1].hist(rng.normal(0, 1, 10_000), bins=40, color="#F58518", alpha=0.85)
axes[1].set_title("标准正态分布 N(0, 1)")
for ax in axes:
    ax.set_ylabel("频数")
plt.tight_layout()
plt.show()


### 2.2 AI 中的权重初始化：Xavier 与 He

神经网络的权重不能随便初始化得太大或太小：

- 太大：激活值和梯度可能爆炸
- 太小：信号层层衰减，学习很慢

常见经验规则：

- Xavier：标准差约为 `sqrt(2 / (n_in + n_out))`，常用于 tanh/sigmoid
- He：标准差约为 `sqrt(2 / n_in)`，常用于 ReLU


In [ ]:
rng = np.random.default_rng(42)
n_in, n_out = 784, 256
xavier = rng.normal(0, np.sqrt(2 / (n_in + n_out)), size=(n_in, n_out))
he = rng.normal(0, np.sqrt(2 / n_in), size=(n_in, n_out))

print("Xavier shape:", xavier.shape, "mean/std:", xavier.mean(), xavier.std())
print("He     shape:", he.shape, "mean/std:", he.mean(), he.std())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(xavier.ravel(), bins=60, alpha=0.65, label="Xavier")
ax.hist(he.ravel(), bins=60, alpha=0.65, label="He")
ax.set_title("权重初始化分布对比")
ax.set_xlabel("权重值")
ax.set_ylabel("频数")
ax.legend()
plt.show()


## 3. 索引与切片：从数组中取你真正需要的部分

索引不是语法细节，而是数据处理的核心：

- 取某一行：一个样本
- 取某一列：一个特征
- 取一个子矩阵：一个 batch 或一组特征
- 用布尔条件筛选：找出满足条件的样本
- 花式索引：按指定位置批量取值


In [ ]:
data = np.array([[ 1,  2,  3,  4],
                 [ 5,  6,  7,  8],
                 [ 9, 10, 11, 12]])

print("data =\n", data)
print("data[0, 0]      ->", data[0, 0])
print("data[-1, -1]    ->", data[-1, -1])
print("data[0, :]      ->", data[0, :], "  # 第 0 行")
print("data[:, 1]      ->", data[:, 1], "  # 第 1 列")
print("data[0:2, 1:3]  ->\n", data[0:2, 1:3])


In [ ]:
# 用颜色标出一个切片 data[0:2, 1:3]
selected = np.zeros_like(data, dtype=float)
selected[0:2, 1:3] = 1

fig, ax = plt.subplots(figsize=(6, 3))
ax.imshow(selected, cmap="Oranges", vmin=0, vmax=1)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, str(data[i, j]), ha="center", va="center", fontsize=12)
ax.set_title("切片 data[0:2, 1:3]：前两行 + 第 1 到 2 列")
ax.set_xticks(range(data.shape[1]))
ax.set_yticks(range(data.shape[0]))
plt.show()


### 3.1 布尔索引：用条件筛选数据

布尔索引会先生成一个同形状的 `True/False` 数组，再用它筛选元素。它非常适合数据清洗和异常值处理。


In [ ]:
mask = data > 5
print("data > 5 得到布尔 mask:\n", mask)
print("筛选结果 data[data > 5]:", data[mask])

# 也可以把满足条件的值替换掉，不改变原数组时用 np.where
clipped_by_where = np.where(data > 8, 8, data)
print("\n把大于 8 的值限制为 8:\n", clipped_by_where)


In [ ]:
# 示例：筛选高分样本
scores = np.array([58, 72, 91, 67, 88, 95, 43])
names = np.array(["A", "B", "C", "D", "E", "F", "G"])
passed = scores >= 60
excellent = scores >= 85

print("及格学生:", names[passed], scores[passed])
print("优秀学生:", names[excellent], scores[excellent])


### 3.2 花式索引：一次取多个指定位置

花式索引容易被误解。`data[[0, 2], [1, 3]]` 不是取第 0、2 行和第 1、3 列组成的子矩阵，而是取两个坐标：

- `(0, 1)`
- `(2, 3)`

如果要取“行集合 × 列集合”的网格，可以用 `np.ix_`。


In [ ]:
rows = [0, 2]
cols = [1, 3]
print("成对取坐标 data[[0,2], [1,3]]:", data[rows, cols])
print("网格取子矩阵 data[np.ix_([0,2], [1,3])]:\n", data[np.ix_(rows, cols)])


## 4. 向量化：让 NumPy 一次处理整个数组

向量化的核心思想：把“对每个元素做同一件事”的循环，改写成“对整个数组做一次运算”。

这通常带来两个好处：

1. 代码更短，更接近数学公式
2. 底层由 C/BLAS 等高性能实现执行，减少 Python 解释器循环开销


In [ ]:
# 小规模例子：逐元素运算
x = np.array([1, 2, 3, 4])
y = np.array([10, 20, 30, 40])

print("x + y      =", x + y)
print("x * y      =", x * y, "  # 逐元素相乘")
print("x ** 2     =", x ** 2)
print("sqrt(x)    =", np.sqrt(x))
print("exp(x)     =", np.exp(x))
print("log(x)     =", np.log(x))


In [ ]:
# 速度对比：Python 循环 vs NumPy 向量化
rng = np.random.default_rng(42)
n = 1_000_000
a = rng.normal(size=n)
b = rng.normal(size=n)

start = time.perf_counter()
c_loop = [a[i] + b[i] for i in range(n)]
loop_time = time.perf_counter() - start

start = time.perf_counter()
c_np = a + b
np_time = time.perf_counter() - start

print(f"数组大小: {n:,}")
print(f"Python 循环: {loop_time:.4f} 秒")
print(f"NumPy 向量化: {np_time:.6f} 秒")
print(f"加速比约: {loop_time / np_time:.0f}x")
print("结果一致:", np.allclose(c_loop, c_np))

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(["Python 循环", "NumPy 向量化"], [loop_time, np_time], color=["#E45756", "#54A24B"])
ax.set_ylabel("运行时间（秒，越低越好）")
ax.set_title("向量化速度对比")
plt.show()


### 4.1 向量化不只是加法：通用函数 ufunc

`np.sqrt`、`np.exp`、`np.log`、`np.maximum` 这类函数叫 ufunc（universal function）。它们会逐元素作用在数组上，并尽量使用高效底层实现。


In [ ]:
xs = np.linspace(-4, 4, 400)
sigmoid_curve = 1 / (1 + np.exp(-xs))
relu_curve = np.maximum(0, xs)
tanh_curve = np.tanh(xs)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(xs, sigmoid_curve, label="sigmoid")
ax.plot(xs, relu_curve, label="ReLU")
ax.plot(xs, tanh_curve, label="tanh")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("用 NumPy 向量化计算常见激活函数")
ax.legend()
plt.show()


## 5. 广播机制：让不同形状自动对齐

广播（broadcasting）可以把较小数组“看起来扩展”为较大形状，而不真的复制数据。

规则从最后一个维度开始比较：

1. 两个维度相等：可以广播
2. 其中一个维度是 1：可以扩展到另一个维度
3. 否则：报错

典型例子：

- `(batch, features) + (features,)`：给每个样本加同一个 bias
- `(n_samples, n_features) - (n_features,)`：每个特征减去自己的均值


In [ ]:
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])
row_bias = np.array([10, 20, 30])
col_bias = np.array([[100], [200], [300]])

print("matrix shape:", matrix.shape)
print("row_bias shape:", row_bias.shape)
print("matrix + row_bias:\n", matrix + row_bias)
print("\ncol_bias shape:", col_bias.shape)
print("matrix + col_bias:\n", matrix + col_bias)


In [ ]:
# 可视化广播：行向量加到每一行
base = np.arange(1, 10).reshape(3, 3)
bias = np.array([10, 20, 30])
result = base + bias

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, arr, title in zip(axes, [base, np.tile(bias, (3, 1)), result], ["矩阵", "广播后的行向量", "相加结果"]):
    ax.imshow(arr, cmap="YlGnBu")
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            ax.text(j, i, str(arr[i, j]), ha="center", va="center")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("(3,3) + (3,) → (3,3)")
plt.tight_layout()
plt.show()


### 5.1 广播实战：特征标准化

标准化是机器学习最常见的数据预处理之一：

$$
 x' = \frac{x - \mu}{\sigma}
$$

对二维数据 `(样本数, 特征数)`，我们通常沿着 `axis=0` 计算每个特征的均值和标准差，得到形状 `(特征数,)`，然后利用广播对每一行应用同一组均值和标准差。


In [ ]:
# 5 个样本，3 个特征：身高(cm)、体重(kg)、年龄
X = np.array([
    [170, 65, 25],
    [165, 55, 30],
    [180, 80, 22],
    [175, 70, 28],
    [160, 50, 35],
], dtype=float)
feature_names = ["身高", "体重", "年龄"]

mean = X.mean(axis=0)
std = X.std(axis=0)
X_scaled = (X - mean) / std

print("原始数据:\n", X)
print("每个特征均值:", mean)
print("每个特征标准差:", std)
print("\n标准化后:\n", X_scaled.round(3))
print("新均值:", X_scaled.mean(axis=0).round(10))
print("新标准差:", X_scaled.std(axis=0).round(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].boxplot([X[:, i] for i in range(X.shape[1])], labels=feature_names)
axes[0].set_title("标准化前：不同特征尺度差异明显")
axes[0].set_ylabel("原始数值")

axes[1].boxplot([X_scaled[:, i] for i in range(X_scaled.shape[1])], labels=feature_names)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("标准化后：都围绕 0，标准差约为 1")
axes[1].set_ylabel("标准化数值")
plt.tight_layout()
plt.show()


### 5.2 广播错误如何读？

如果最后一维对不上，并且没有维度是 1，NumPy 会报 shape 相关错误。

例如 `(3, 4) + (3,)` 失败，因为从最后一维比较是 `4` vs `3`。


In [ ]:
bad_a = np.zeros((3, 4))
bad_b = np.ones((3,))
try:
    bad_a + bad_b
except ValueError as e:
    print("广播失败示例:")
    print(e)

# 修复：如果想按行加，需要把 (3,) 变成 (3,1)
fixed = bad_a + bad_b[:, None]
print("\n修复后 shape:", fixed.shape)


## 6. 聚合函数与 axis：到底沿哪个方向算？

`axis` 是 NumPy 初学者最容易混乱的参数。

对二维数组：

- `axis=0`：压缩第 0 轴，也就是“沿着行往下看”，结果按列统计
- `axis=1`：压缩第 1 轴，也就是“沿着列横向看”，结果按行统计

一句话：`axis` 指的是“被消掉的轴”。


In [ ]:
arr = np.array([[1, 5, 3],
                [4, 2, 6]])
print("arr =\n", arr)
print("sum()        =", arr.sum())
print("sum(axis=0) =", arr.sum(axis=0), "  # 每列求和")
print("sum(axis=1) =", arr.sum(axis=1), "  # 每行求和")
print("mean()       =", arr.mean())
print("max(axis=1) =", arr.max(axis=1))
print("argmax()     =", arr.argmax(), "  # 展平后的索引")
print("argmax(axis=1)=", arr.argmax(axis=1), " # 每行最大值的列索引")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

# axis=0
axes[0].imshow(arr, cmap="Purples")
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        axes[0].text(j, i, str(arr[i,j]), ha="center", va="center")
for j, s in enumerate(arr.sum(axis=0)):
    axes[0].text(j, 2.0, f"↓ {s}", ha="center", va="center", color="crimson", fontsize=12)
axes[0].set_title("axis=0：按列聚合")
axes[0].set_xlim(-0.5, 2.5); axes[0].set_ylim(2.3, -0.5)
axes[0].set_xticks([]); axes[0].set_yticks([])

# axis=1
axes[1].imshow(arr, cmap="Greens")
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        axes[1].text(j, i, str(arr[i,j]), ha="center", va="center")
for i, s in enumerate(arr.sum(axis=1)):
    axes[1].text(3.0, i, f"→ {s}", ha="center", va="center", color="crimson", fontsize=12)
axes[1].set_title("axis=1：按行聚合")
axes[1].set_xlim(-0.5, 3.4); axes[1].set_ylim(1.5, -0.5)
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout()
plt.show()


### 6.1 形状操作：reshape、transpose、拼接

形状操作不改变数学意义时很方便，但也很危险：

- `reshape`：重新解释同一批元素的形状，元素总数必须不变
- `-1`：让 NumPy 自动推断该维度大小
- `.T`：二维矩阵转置
- `vstack/hstack/concatenate`：拼接数组

深度学习中，很多张量会在 `(batch, height, width, channels)`、`(batch, features)` 等形状之间转换。


In [ ]:
arr = np.arange(12)
print("原始:", arr, "shape=", arr.shape)
print("reshape(3,4):\n", arr.reshape(3, 4))
print("reshape(2, -1):\n", arr.reshape(2, -1))
print("reshape(3,4).T:\n", arr.reshape(3, 4).T)

x1 = np.array([[1, 2], [3, 4]])
x2 = np.array([[5, 6], [7, 8]])
print("\nvstack:\n", np.vstack([x1, x2]))
print("hstack:\n", np.hstack([x1, x2]))


In [ ]:
# 图片展平示意：4x4 灰度图 -> 16 维特征向量
image = np.array([
    [0.0, 0.2, 0.8, 1.0],
    [0.1, 0.7, 0.9, 0.6],
    [0.0, 0.4, 0.8, 0.3],
    [0.0, 0.1, 0.2, 0.0],
])
flat = image.reshape(-1)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("4x4 灰度图")
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[1].bar(np.arange(flat.size), flat, color="#4C78A8")
axes[1].set_title("reshape(-1) 后的 16 维向量")
axes[1].set_xlabel("特征索引")
axes[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 7. 矩阵乘法：从特征到模型输出

NumPy 中的 `*` 是逐元素相乘，不是矩阵乘法。矩阵乘法使用 `@` 或 `np.matmul`。

在神经网络中，一层线性变换通常写成：

$$
Y = XW + b
$$

其中：

- `X`：输入 batch，形状 `(batch_size, n_features)`
- `W`：权重矩阵，形状 `(n_features, n_outputs)`
- `b`：偏置，形状 `(n_outputs,)`
- `Y`：输出，形状 `(batch_size, n_outputs)`


In [ ]:
rng = np.random.default_rng(42)
X_batch = rng.normal(size=(5, 3))      # 5 个样本，每个 3 个特征
W = rng.normal(size=(3, 2))            # 从 3 维映射到 2 维
b = np.array([0.5, -0.5])              # 广播到每个样本
Y = X_batch @ W + b

print("X_batch shape:", X_batch.shape)
print("W shape      :", W.shape)
print("b shape      :", b.shape)
print("Y shape      :", Y.shape)
print("\nY =\n", Y.round(3))


In [ ]:
# 可视化一个二维线性变换：旋转 + 缩放
points = np.array([[0,0], [1,0], [1,1], [0,1], [0,0]], dtype=float)
A = np.array([[1.2, 0.5],
              [-0.3, 0.9]])
transformed = points @ A.T

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(points[:,0], points[:,1], "o-", label="原始单位方形")
ax.plot(transformed[:,0], transformed[:,1], "o-", label="线性变换后")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_aspect("equal", adjustable="box")
ax.set_title("矩阵乘法 = 对空间做线性变换")
ax.legend()
plt.show()


### 7.1 从零实现 Sigmoid、ReLU 与 Softmax

激活函数也是数组函数。关键是：同一个函数可以同时处理标量、向量、矩阵甚至更高维数组。

Softmax 常用于多分类，它把 logits 转成概率分布。实现时要先减去最大值，避免 `exp` 溢出：

$$
\mathrm{softmax}(z_i)=\frac{e^{z_i - \max(z)}}{\sum_j e^{z_j - \max(z)}}
$$


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def softmax(x, axis=-1):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

x = np.array([-3, -1, 0, 1, 3])
print("x:", x)
print("sigmoid(x):", sigmoid(x).round(4))
print("relu(x):", relu(x))

logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)
print("\nlogits:", logits)
print("softmax:", probs.round(4))
print("sum:", probs.sum())


In [ ]:
# 温度参数会改变 softmax 的“尖锐程度”
logits = np.array([2.0, 1.0, 0.1])
temperatures = [0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, T in zip(axes, temperatures):
    p = softmax(logits / T)
    ax.bar(["类0", "类1", "类2"], p, color="#72B7B2")
    ax.set_ylim(0, 1)
    ax.set_title(f"T={T}\n{p.round(3)}")
axes[0].set_ylabel("概率")
plt.suptitle("Softmax 温度：越低越自信，越高越平滑")
plt.tight_layout()
plt.show()


### 7.2 一个极简分类器：`X @ W + b -> softmax`

下面把前面的知识串起来：

1. `X` 是一个 batch 的输入特征
2. `W` 和 `b` 是模型参数
3. `logits = X @ W + b` 得到每个类别的未归一化分数
4. `softmax(logits)` 得到每个类别的概率
5. `argmax` 得到预测类别


In [ ]:
rng = np.random.default_rng(7)
X = rng.normal(size=(6, 4))       # 6 个样本，4 个特征
W = rng.normal(scale=0.5, size=(4, 3))  # 3 个类别
b = np.array([0.1, -0.2, 0.0])

logits = X @ W + b
probs = softmax(logits, axis=1)
pred = np.argmax(probs, axis=1)

print("X shape     :", X.shape)
print("W shape     :", W.shape)
print("logits shape:", logits.shape)
print("probs shape :", probs.shape)
print("每行概率和  :", probs.sum(axis=1).round(6))
print("预测类别    :", pred)
print("\n前两个样本概率:\n", probs[:2].round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
im = ax.imshow(probs, cmap="viridis", vmin=0, vmax=1)
for i in range(probs.shape[0]):
    for j in range(probs.shape[1]):
        ax.text(j, i, f"{probs[i,j]:.2f}", ha="center", va="center", color="white" if probs[i,j] > 0.5 else "black")
ax.set_title("极简分类器输出：每个样本属于每个类别的概率")
ax.set_xlabel("类别")
ax.set_ylabel("样本")
ax.set_xticks(range(3), ["类0", "类1", "类2"])
ax.set_yticks(range(6), [f"样本{i}" for i in range(6)])
plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()


## 8. 常见误区与检查清单

### 常见误区

| 误区 | 正确理解 |
|---|---|
| `*` 是矩阵乘法 | `*` 是逐元素乘法，矩阵乘法用 `@` |
| `axis=0` 表示按行求和 | `axis=0` 表示消掉第 0 轴，结果是按列统计 |
| 广播会真的复制数组 | 广播通常是“视图式”的逻辑扩展，不一定复制内存 |
| `reshape` 可以随便改形状 | 元素总数必须一致，且语义要合理 |
| 随机数不设种子也没关系 | 教程、实验复现、debug 时最好固定种子 |
| softmax 直接 `np.exp(x)` | 大 logits 会溢出，先减 `max` 更稳定 |

### Shape 调试建议

遇到 NumPy/AI 代码 bug 时，先打印：

```python
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
```

再逐步检查：

1. 矩阵乘法内维是否相等：`(m, n) @ (n, k)`
2. 广播是否从最后一维开始匹配
3. 聚合后是否需要 `keepdims=True`
4. `reshape` 前后元素总数是否一致


In [ ]:
# 一个常用的小工具：快速打印多个数组的形状
def shape_report(**arrays):
    for name, arr in arrays.items():
        arr = np.asarray(arr)
        print(f"{name:>10}: shape={arr.shape}, ndim={arr.ndim}, dtype={arr.dtype}")

shape_report(X=X, W=W, b=b, logits=logits, probs=probs, pred=pred)


## 9. 本节总结与练习

### 关键要点

1. `ndarray = 数据 + shape + dtype`，shape 是 AI 编程的第一调试对象
2. 向量化让代码更接近数学表达，也通常更快
3. 广播让 `(batch, features) + (features,)` 这类批量计算非常自然
4. `axis` 表示“被压缩/消掉的轴”，不是简单的“行/列”中文翻译
5. `@`、`softmax`、标准化、权重初始化可以组合成最小的 AI 计算流程

### 与 AI 的连接

如果把 PyTorch / TensorFlow 的张量先看成“支持自动求导和 GPU 的 NumPy 数组”，很多概念会更容易理解：

- batch：二维或更高维数组的第 0 轴
- feature：通常是最后一维或某个通道维
- weight：参与矩阵乘法的参数数组
- bias：依靠广播加到每个样本上
- activation：逐元素 ufunc

### 练习

1. 创建一个形状为 `(100, 3)` 的随机数据集，分别对 3 个特征做标准化。
2. 手写一个函数 `min_max_scale(X)`，把每列缩放到 `[0, 1]`。
3. 构造 `X.shape=(8, 5)`、`W.shape=(5, 4)`、`b.shape=(4,)`，计算 `softmax(X @ W + b)`。
4. 故意制造一个广播错误，读懂报错信息后修复它。
5. 用 `matplotlib` 画出 `sigmoid`、`tanh`、`relu` 的曲线并比较它们的输出范围。
